In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

from pathlib import Path
from tqdm import tqdm
from collections import Counter

In [51]:
# All the input paths
inspire_path = Path("/home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3")
ops_path = inspire_path / "operations.csv"
labs_path = inspire_path / "labs.csv"
vitals_path = inspire_path / "vitals.csv"
ward_vitals_path = inspire_path / "ward_vitals.csv"
combined_path = inspire_path / "combined_data.csv"
combined_cleaned_path = inspire_path / "combined_cleaned_data.csv"
pca_path = inspire_path / "pca_data.csv"
diagnosis_path = inspire_path / "diagnosis.csv"

preopdata_file = "/home/server/Projects/data/AKI/preop_data.csv"
preopdata_file_nidhir = "/home/server/Projects/data/AKI/preop_data_nidhir.csv"
aki_data = "/home/server/Projects/data/AKI/aki_data.csv"

# Output path
output_path = Path("/home/server/Projects/data/AKI")

In [90]:
df = pd.read_csv(ops_path)
df_ops = df.copy()

In [55]:
Counter(df['antype'])

Counter({0: 102790, 1: 28008, 'Regional': 162})

In [113]:
df_ops = df_ops.drop(df_ops[df_ops['antype'] == 'Regional'].index)
df_ops.loc[df_ops['antype'] == 'General', 'antype'] = 0     #replace antypes with numbers, after removing rows with regional set as antype
df_ops.loc[df_ops['antype'] == 'MAC', 'antype'] = 1
df_ops.loc[df_ops['antype'] == 'Neuraxial', 'antype'] = 1

col = df_ops.columns.get_loc('department')                  #don't want to just add encodings to end of dataframe, so insert it where department used to be
num_cols_added = len(Counter(df_ops['department']))
ops_general = pd.get_dummies(df_ops, columns=['department'])
ops_gen_cols_to_keep = ['op_id', 'subject_id', 'antype']
for column_idx in range(col, col + num_cols_added):
    department_name = ops_general.columns[-1]
    ops_gen_cols_to_keep.append(department_name)
    ops_general.insert(column_idx, department_name, ops_general.pop(department_name))

ops_general

,op_id,subject_id,hadm_id,case_id,opdate,age,sex,weight,height,race,...,admission_time,discharge_time,anstart_time,anend_time,cpbon_time,cpboff_time,icuin_time,icuout_time,inhosp_death_time,allcause_death_time
0,484069807,178742874,229842382,NaN,0,30,F,50.0,155.0,Asian,...,0,7195,1120.0,1235.0,NaN,NaN,NaN,NaN,NaN,NaN
1,446270725,158995752,257857903,NaN,0,70,M,45.0,170.0,Asian,...,0,70555,1345.0,1540.0,NaN,NaN,1550.0,19595.0,69860.0,106560.0
2,406892271,108553242,200664328,NaN,61920,50,F,70.0,165.0,Asian,...,0,178555,62170.0,62370.0,NaN,NaN,NaN,NaN,NaN,718560.0
3,478413008,133278262,277235295,NaN,0,35,F,55.0,NaN,Asian,...,0,5755,215.0,340.0,NaN,NaN,NaN,NaN,NaN,NaN
4,468516791,116924034,299190423,NaN,17280,45,F,45.0,150.0,Asian,...,0,25915,17950.0,18070.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130955,449124488,138484174,228449654,NaN,4999680,50,F,60.0,160.0,Asian,...,4996800,5012635,5000390.0,5000570.0,NaN,NaN,NaN,NaN,NaN,NaN
130956,461252752,126772283,273139806,NaN,2880,70,F,55.0,160.0,Asian,...,0,7195,3355.0,3430.0,NaN,NaN,NaN,NaN,NaN,NaN
130957,471834474,144363433,275833861,NaN,2880,65,F,50.0,150.0,Asian,...,0,11515,3595.0,3670.0,NaN,NaN,NaN,NaN,NaN,NaN
130958,419787421,195835964,293939099,NaN,12960,85,M,75.0,170.0,Asian,...,0,27355,13780.0,13950.0,NaN,NaN,13955.0,15120.0,NaN,613440.0


In [102]:
df




,op_id,subject_id,hadm_id,case_id,opdate,age,sex,weight,height,race,...,admission_time,discharge_time,anstart_time,anend_time,cpbon_time,cpboff_time,icuin_time,icuout_time,inhosp_death_time,allcause_death_time
0,484069807,178742874,229842382,NaN,0,30,F,50.0,155.0,Asian,...,0,7195,1120.0,1235.0,NaN,NaN,NaN,NaN,NaN,NaN
1,446270725,158995752,257857903,NaN,0,70,M,45.0,170.0,Asian,...,0,70555,1345.0,1540.0,NaN,NaN,1550.0,19595.0,69860.0,106560.0
2,406892271,108553242,200664328,NaN,61920,50,F,70.0,165.0,Asian,...,0,178555,62170.0,62370.0,NaN,NaN,NaN,NaN,NaN,718560.0
3,478413008,133278262,277235295,NaN,0,35,F,55.0,NaN,Asian,...,0,5755,215.0,340.0,NaN,NaN,NaN,NaN,NaN,NaN
4,468516791,116924034,299190423,NaN,17280,45,F,45.0,150.0,Asian,...,0,25915,17950.0,18070.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130955,449124488,138484174,228449654,NaN,4999680,50,F,60.0,160.0,Asian,...,4996800,5012635,5000390.0,5000570.0,NaN,NaN,NaN,NaN,NaN,NaN
130956,461252752,126772283,273139806,NaN,2880,70,F,55.0,160.0,Asian,...,0,7195,3355.0,3430.0,NaN,NaN,NaN,NaN,NaN,NaN
130957,471834474,144363433,275833861,NaN,2880,65,F,50.0,150.0,Asian,...,0,11515,3595.0,3670.0,NaN,NaN,NaN,NaN,NaN,NaN
130958,419787421,195835964,293939099,NaN,12960,85,M,75.0,170.0,Asian,...,0,27355,13780.0,13950.0,NaN,NaN,13955.0,15120.0,NaN,613440.0


In [114]:
from pathlib import Path
import pandas as pd

def nprint(string):
    print("="*25, string, "="*25)

# All the input paths
inspire_path = Path("/home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3")
labs_path = inspire_path / "labs.csv"
vitals_path = inspire_path / "vitals.csv"
ops_path = inspire_path / "operations.csv"

preopdata_file = "/home/server/Projects/data/AKI/preop_data.csv"
preopdata_file_nidhir = "/home/server/Projects/data/AKI/preop_data_nidhir.csv"

nprint("starting")
df_labs = pd.read_csv(labs_path)
df_labs["chart_time"] = df_labs["chart_time"].astype(float)
df_vitals = pd.read_csv(vitals_path)
df_preop = pd.read_csv(preopdata_file)
df_ops = pd.read_csv(ops_path)
nprint("finished reading csvs")

df_preop = df_preop[df_preop["asa"] < 6]
df_preop = df_preop[df_preop["age"] >= 18]
df_preop = df_preop.dropna(subset="opend_time")
df_preop = df_preop.dropna(subset="opstart_time")
df_preop["op_len"] = df_preop["opend_time"] - df_preop["opstart_time"]

#change
df_ops = df_ops.drop(df_ops[df_ops['antype'] == 'Regional'].index)
df_ops.loc[df_ops['antype'] == 'General', 'antype'] = 0     #replace antypes with numbers, after removing rows with regional set as antype
df_ops.loc[df_ops['antype'] == 'MAC', 'antype'] = 1
df_ops.loc[df_ops['antype'] == 'Neuraxial', 'antype'] = 1

col = df_ops.columns.get_loc('department')                  #don't want to just add encodings to end of dataframe, so insert it where department used to be
num_cols_added = len(Counter(df_ops['department']))
ops_general = pd.get_dummies(df_ops, columns=['department'])
ops_gen_cols_to_keep = ['op_id', 'subject_id', 'antype']
for column_idx in range(col, col + num_cols_added):
    department_name = ops_general.columns[-1]
    ops_gen_cols_to_keep.append(department_name)
    ops_general.insert(column_idx, department_name, ops_general.pop(department_name))
df_preop = pd.merge(df_preop, ops_general[ops_gen_cols_to_keep], on=['op_id', 'subject_id'], how='inner')

# ops_general = df_ops[df_ops['antype'] == 'General']
# df_preop = pd.merge(df_preop, ops_general[['op_id', 'subject_id']], on=['op_id', 'subject_id'], how='inner')

nprint("finished basic filtering")

item_names = [
    "total_protein",
    "sodium",
    "potassium",
    "platelet",
    "glucose",
    "wbc",
    "alt",
    "chloride",
    "lymphocyte",
    "phosphorus",
    "albumin",
    "fibrinogen",
    "creatinine",
    "ptinr",
    "total_bilirubin",
    "alp",
    "aptt",
    "calcium",
    "bun",
    "ast",
    "crp",
    "hb",
    "hct",
    "seg"
]

for item_name in item_names:
    df_preop = pd.merge_asof(df_preop.sort_values('opstart_time'), 
                    df_labs.loc[df_labs['item_name'] == item_name].sort_values('chart_time'), # grab rows w the item name we want and sort by chart_time
                    left_on='opstart_time', right_on='chart_time', by='subject_id',           # chooses row in df_labs w greatest chart_time that is still less than opstart_time and matches subject_id
                    tolerance=90 * 24 * 60, suffixes=('', '_'))                               # 90 day tolerance
    df_preop.drop(columns=['chart_time', 'item_name'], inplace=True)
    df_preop.rename(columns={'value':f'preop_{item_name}'}, inplace=True)
nprint("finished getting preop data from time series")

df_creatinine = df_labs[df_labs['item_name'] == 'creatinine']
df_merge = pd.merge(df_preop, df_creatinine, on='subject_id', suffixes=('_preop', '_lab'))
df_merge_filtered = df_merge[
    (df_merge['chart_time'] > df_merge['opend_time']) &
    (df_merge['chart_time'] < (df_merge['opend_time'] + 2*24*60))
]
max_creatinine = (
    df_merge_filtered.groupby(['subject_id', 'op_id'])['value']
    .max()
    .reset_index()
    .rename(columns={'value': 'postop_creatinine'})
)
df_preop = pd.merge(df_preop, max_creatinine, on=['subject_id', 'op_id'], how='inner')
nprint("finished getting postop data from time series")

df_preop = df_preop[df_preop["preop_creatinine"] <= 4.5]

prefixes_to_exclude = ["10", "0TY", "B50", "B51"]
mask = df_ops["icd10_pcs"].astype(str).str.startswith(tuple(prefixes_to_exclude))
ops_to_exclude = df_ops.loc[mask, "op_id"]
df_preop = df_preop[~df_preop["op_id"].isin(ops_to_exclude)]
nprint("finished filtering out some procedure prefixes")

df_preop["aki"] = df_preop["postop_creatinine"] - df_preop["preop_creatinine"]
nprint("calculated aki")

========================= starting =========================
========================= finished reading csvs =========================
========================= finished basic filtering =========================
========================= finished getting preop data from time series =========================
========================= finished getting postop data from time series =========================
========================= finished filtering out some procedure prefixes =========================
========================= calculated aki =========================


In [115]:
df_preop

,op_id,subject_id,age,sex,height,weight,asa,emop,opstart_time,opend_time,...,preop_aptt,preop_calcium,preop_bun,preop_ast,preop_crp,preop_hb,preop_hct,preop_seg,postop_creatinine,aki
0,446345798,149047291,60,M,170.0,65.0,4.0,1,20.0,230.0,...,31.2,7.5,17.0,35.0,NaN,7.6,22.0,86.8,0.92,0.24
1,468145039,130014860,45,M,NaN,65.0,1.0,1,50.0,120.0,...,34.7,9.1,12.0,31.0,2.25,15.6,46.0,89.7,1.05,0.17
3,403543518,164935053,45,M,180.0,65.0,1.0,1,65.0,300.0,...,25.1,9.3,18.0,108.0,0.19,14.2,40.4,92.0,1.33,0.41
4,490288058,143981214,75,M,NaN,NaN,3.0,1,65.0,430.0,...,32.3,8.7,36.0,14.0,4.58,12.2,33.5,73.7,1.69,-1.35
6,404463214,174118101,35,F,175.0,55.0,3.0,1,70.0,100.0,...,24.0,8.8,7.0,18.0,0.06,11.6,33.5,92.0,0.53,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60340,403983493,107333634,70,F,150.0,50.0,2.0,0,5076525.0,5076605.0,...,30.6,9.6,12.0,17.0,0.02,13.7,40.4,69.6,0.61,-0.10
60341,434627934,106196053,65,M,165.0,70.0,3.0,0,5079805.0,5080010.0,...,32.3,8.7,19.5,13.0,0.31,15.0,33.5,77.8,0.77,0.00
60342,485921156,132778234,70,F,155.0,65.0,2.0,0,5081095.0,5081230.0,...,32.3,8.9,21.0,19.0,0.19,12.9,36.1,49.5,0.65,-0.09
60343,452652040,117366461,70,M,175.0,70.0,2.0,0,5089780.0,5089830.0,...,NaN,9.8,18.0,31.0,0.06,15.0,40.4,63.2,0.92,-0.06


In [ ]:
#drop booking case length

In [108]:
from pathlib import Path
import pandas as pd

def nprint(string):
    print("="*25, string, "="*25)

# All the input paths
inspire_path = Path("/home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3")
labs_path = inspire_path / "labs.csv"
vitals_path = inspire_path / "vitals.csv"
ops_path = inspire_path / "operations.csv"

preopdata_file = "/home/server/Projects/data/AKI/preop_data.csv"
preopdata_file_nidhir = "/home/server/Projects/data/AKI/preop_data_nidhir.csv"

nprint("starting")
df_labs = pd.read_csv(labs_path)
df_labs["chart_time"] = df_labs["chart_time"].astype(float)
df_vitals = pd.read_csv(vitals_path)
df_preop = pd.read_csv(preopdata_file)
df_ops = pd.read_csv(ops_path)
nprint("finished reading csvs")

df_preop = df_preop[df_preop["asa"] < 6]
df_preop = df_preop[df_preop["age"] >= 18]
df_preop = df_preop.dropna(subset="opend_time")
df_preop = df_preop.dropna(subset="opstart_time")
df_preop["op_len"] = df_preop["opend_time"] - df_preop["opstart_time"]


========================= starting =========================
========================= finished reading csvs =========================


In [109]:
df_ops = df_ops.drop(df_ops[df_ops['antype'] == 'Regional'].index)
df_ops.loc[df_ops['antype'] == 'General', 'antype'] = 0     #replace antypes with numbers, after removing rows with regional set as antype
df_ops.loc[df_ops['antype'] == 'MAC', 'antype'] = 1
df_ops.loc[df_ops['antype'] == 'Neuraxial', 'antype'] = 1

col = df_ops.columns.get_loc('department')                  #don't want to just add encodings to end of dataframe, so insert it where department used to be
num_cols_added = len(Counter(df_ops['department']))
ops_general = pd.get_dummies(df_ops, columns=['department'])
for column_idx in range(col, col + num_cols_added):
    department_name = ops_general.columns[-1]
    ops_general.insert(column_idx, department_name, ops_general.pop(department_name))

In [110]:
ops_general

,op_id,subject_id,hadm_id,case_id,opdate,age,sex,weight,height,race,...,admission_time,discharge_time,anstart_time,anend_time,cpbon_time,cpboff_time,icuin_time,icuout_time,inhosp_death_time,allcause_death_time
0,484069807,178742874,229842382,NaN,0,30,F,50.0,155.0,Asian,...,0,7195,1120.0,1235.0,NaN,NaN,NaN,NaN,NaN,NaN
1,446270725,158995752,257857903,NaN,0,70,M,45.0,170.0,Asian,...,0,70555,1345.0,1540.0,NaN,NaN,1550.0,19595.0,69860.0,106560.0
2,406892271,108553242,200664328,NaN,61920,50,F,70.0,165.0,Asian,...,0,178555,62170.0,62370.0,NaN,NaN,NaN,NaN,NaN,718560.0
3,478413008,133278262,277235295,NaN,0,35,F,55.0,NaN,Asian,...,0,5755,215.0,340.0,NaN,NaN,NaN,NaN,NaN,NaN
4,468516791,116924034,299190423,NaN,17280,45,F,45.0,150.0,Asian,...,0,25915,17950.0,18070.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130955,449124488,138484174,228449654,NaN,4999680,50,F,60.0,160.0,Asian,...,4996800,5012635,5000390.0,5000570.0,NaN,NaN,NaN,NaN,NaN,NaN
130956,461252752,126772283,273139806,NaN,2880,70,F,55.0,160.0,Asian,...,0,7195,3355.0,3430.0,NaN,NaN,NaN,NaN,NaN,NaN
130957,471834474,144363433,275833861,NaN,2880,65,F,50.0,150.0,Asian,...,0,11515,3595.0,3670.0,NaN,NaN,NaN,NaN,NaN,NaN
130958,419787421,195835964,293939099,NaN,12960,85,M,75.0,170.0,Asian,...,0,27355,13780.0,13950.0,NaN,NaN,13955.0,15120.0,NaN,613440.0


In [111]:
df_preop = pd.merge(df_preop, ops_general[['op_id', 'subject_id']], on=['op_id', 'subject_id'], how='inner')
df_preop

,op_id,subject_id,age,sex,height,weight,asa,emop,opstart_time,opend_time,inhosp_death_time,allcause_death_time,BSA,BMI,booking_case_length,num_card_events,last_preop_scr,min_preop_scr,op_len
0,468516791,116924034,45,F,150.0,45.0,1.0,0,17970.0,18070.0,NaN,NaN,1.369306,20.000000,135.0,0,0.71,0.71,100.0
1,493866243,174229093,50,F,160.0,75.0,2.0,0,2020.0,2070.0,NaN,NaN,1.825742,29.296875,95.0,4,0.88,0.77,50.0
2,491416905,153073110,50,F,160.0,60.0,1.0,0,2295.0,2565.0,NaN,NaN,1.632993,23.437500,310.0,0,0.58,0.58,270.0
3,467806534,156776324,35,M,175.0,85.0,2.0,0,85490.0,85895.0,NaN,NaN,2.032718,27.755102,485.0,0,1.33,1.33,405.0
4,471265968,198640844,70,F,150.0,50.0,2.0,0,10630.0,10690.0,NaN,4350240.0,1.443376,22.222222,115.0,0,0.98,0.98,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127178,449124488,138484174,50,F,160.0,60.0,2.0,0,5000410.0,5000570.0,NaN,NaN,1.632993,23.437500,190.0,0,0.71,0.58,160.0
127179,461252752,126772283,70,F,160.0,55.0,2.0,0,3370.0,3430.0,NaN,NaN,1.563472,21.484375,85.0,0,0.68,0.68,60.0
127180,471834474,144363433,65,F,150.0,50.0,2.0,0,3610.0,3665.0,NaN,NaN,1.443376,22.222222,80.0,0,0.65,0.61,55.0
127181,419787421,195835964,85,M,170.0,75.0,4.0,0,13815.0,13945.0,NaN,613440.0,1.881932,25.951557,175.0,1,0.65,0.65,130.0


In [ ]:
saved_ops = df_ops


In [107]:
df_ops = df_ops.drop(df_ops[df_ops['antype'] == 'Regional'].index)
df_ops.loc[df_ops['antype'] == 'General', 'antype'] = 0     #replace antypes with numbers, after removing rows with regional set as antype
df_ops.loc[df_ops['antype'] == 'MAC', 'antype'] = 1
df_ops.loc[df_ops['antype'] == 'Neuraxial', 'antype'] = 1

col = df_ops.columns.get_loc('department')
num_cols_added = len(Counter(df_ops['department']))
ops_general = pd.get_dummies(df_ops, columns=['department'])
for column_idx in range(col, col + num_cols_added):
    department_name = ops_general.columns[-1]
    ops_general.insert(column_idx, department_name, ops_general.pop(department_name))
ops_general

,op_id,subject_id,hadm_id,case_id,opdate,age,sex,weight,height,race,...,admission_time,discharge_time,anstart_time,anend_time,cpbon_time,cpboff_time,icuin_time,icuout_time,inhosp_death_time,allcause_death_time
0,484069807,178742874,229842382,NaN,0,30,F,50.0,155.0,Asian,...,0,7195,1120.0,1235.0,NaN,NaN,NaN,NaN,NaN,NaN
1,446270725,158995752,257857903,NaN,0,70,M,45.0,170.0,Asian,...,0,70555,1345.0,1540.0,NaN,NaN,1550.0,19595.0,69860.0,106560.0
2,406892271,108553242,200664328,NaN,61920,50,F,70.0,165.0,Asian,...,0,178555,62170.0,62370.0,NaN,NaN,NaN,NaN,NaN,718560.0
3,478413008,133278262,277235295,NaN,0,35,F,55.0,NaN,Asian,...,0,5755,215.0,340.0,NaN,NaN,NaN,NaN,NaN,NaN
4,468516791,116924034,299190423,NaN,17280,45,F,45.0,150.0,Asian,...,0,25915,17950.0,18070.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130955,449124488,138484174,228449654,NaN,4999680,50,F,60.0,160.0,Asian,...,4996800,5012635,5000390.0,5000570.0,NaN,NaN,NaN,NaN,NaN,NaN
130956,461252752,126772283,273139806,NaN,2880,70,F,55.0,160.0,Asian,...,0,7195,3355.0,3430.0,NaN,NaN,NaN,NaN,NaN,NaN
130957,471834474,144363433,275833861,NaN,2880,65,F,50.0,150.0,Asian,...,0,11515,3595.0,3670.0,NaN,NaN,NaN,NaN,NaN,NaN
130958,419787421,195835964,293939099,NaN,12960,85,M,75.0,170.0,Asian,...,0,27355,13780.0,13950.0,NaN,NaN,13955.0,15120.0,NaN,613440.0
